## DMS Preembedding
Here we will do the preembedding of the DMS (old) dataset for the ESM model and the BERT ESM-init model. 

--- 
### ESM (CLS embedding)

In [1]:
import os
import sys
import tqdm
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, EsmModel

from pnlp.embedding.tokenizer import ProteinTokenizer
from pnlp.embedding.nlp_embedding import NLPEmbedding
from pnlp.model.bert import BERT

/home/kaetlyn/miniforge3/envs/spike_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Parquet Making
class DMSDataset(Dataset):
    """ DMS virus sequence dataset. """

    def __init__(self, csv_file:str):
        self.df = pd.read_csv(csv_file, header=0, na_filter=False)
        self.max_sequence_length = self.df['sequence'].apply(len).max()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        columns = (
            self.df.iloc[idx]["label"],
            self.df.iloc[idx]["target"],
            self.df.iloc[idx]["sequence"],
            self.df.iloc[idx]["ACE2-binding_affinity"],
            self.df.iloc[idx]["RBD_expression"],
        )
        return columns

class ESM(nn.Module):
    def __init__(self, esm):
        super().__init__()
        self.esm = esm

    def forward(self, tokenized_seqs):
        with torch.set_grad_enabled(self.training):  # Enable gradients, managed by model.eval() or model.train() in epoch_iteration
            esm_last_hidden_state = self.esm(**tokenized_seqs).last_hidden_state # shape: (batch_size, sequence_length, embedding_dim)
            esm_cls_embedding = esm_last_hidden_state[:, 0, :]  # CLS token embedding (sequence-level representations)
        return esm_cls_embedding

def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the ESM model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []

    for _, batch_data in data_iter:
        labels, targets, sequences, binding_scores, expression_scores = batch_data 

        # Add 2 to max_length to account for additional tokens added to beginning and end by ESM
        max_length = dataloader.dataset.max_sequence_length + 2
        tokenized_seqs = tokenizer(sequences, return_tensors='pt', padding='max_length', max_length=max_length).to(device) 
        
        with torch.no_grad():
            embeddings = model(tokenized_seqs) # shape: [batch_size, embedding_dim]

        # Create a DataFrame for the batch
        batch_df = pd.DataFrame({
            "label": labels,
            "target": targets,
            "ACE2-binding_affinity": binding_scores,
            "RBD_expression": expression_scores,
            "embedding": [embedding.cpu().numpy() for embedding in embeddings]  # Convert tensor to numpy array for each embedding
        })

        batch_dataframes.append(batch_df)

    # Concatenate all batch DataFrames into one
    result_df = pd.concat(batch_dataframes, ignore_index=True)
    
    # Save data to a Parquet file
    save_as = csv_file.replace(".csv", "_ESM-CLS-embedded.parquet")
    save_as = save_as.replace("/data/dms", "/data/dms/parquets")
    result_df.to_parquet(save_as, index=False)
    print(f"Data saved to {save_as}")

In [3]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, train_dataloader, device, train_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 1285/1285 [01:50<00:00, 11.60it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_train_ESM-CLS-embedded.parquet


In [4]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, test_dataloader, device, test_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 322/322 [00:19<00:00, 16.48it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_test_ESM-CLS-embedded.parquet


Now check the values in the parquet file:

In [5]:
class DMSEmbeddedDataset(Dataset):
    """ Binding or Expression DMS Embedded Dataset, single target. """
    
    def __init__(self, parquet_file:str, result_tag:str):
        """
        Load from parquet file into pandas:
        - sequence label ('labels'), 
        - 'embedding',
        - binding or expression target,
        """
        try:
            self.full_df = pd.read_parquet(parquet_file, engine='fastparquet')
            self.target = 'ACE2-binding_affinity' if 'binding' in result_tag else 'RBD_expression'
        except (FileNotFoundError, pd.errors.ParserError, Exception) as e:
            print(f"Error reading in .parquet file: {parquet_file}\n{e}", file=sys.stderr)
            sys.exit(1)

    def __len__(self) -> int:
        return len(self.full_df)

    def __getitem__(self, idx):
        # label, embedding, target
        return self.full_df['label'][idx], torch.tensor(self.full_df['embedding'][idx]), self.full_df[self.target][idx]

# Load in the parquets
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
embedded_train_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_train_ESM-CLS-embedded.parquet")
train_parquet_loader = DMSEmbeddedDataset(embedded_train_parquet, "binding")
print("Loaded training binding dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

train_parquet_loader = DMSEmbeddedDataset(embedded_train_parquet, "expression")
print("\nLoaded training expression dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

embedded_test_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_test_ESM-CLS-embedded.parquet")
test_parquet_loader = DMSEmbeddedDataset(embedded_test_parquet, "binding")
print("\nLoaded test binding dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

test_parquet_loader = DMSEmbeddedDataset(embedded_test_parquet, "expressino")
print("\nLoaded test expression dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

Loaded training binding dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 8.62
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 9.78
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 6.0
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.6
SARS-CoV-2-Y91C, torch.Size([320]), 10.364166666666668

Loaded training expression dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 7.45
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.07
SARS-CoV-2-Y91C, torch.Size([320]), 9.300833333333332

Loaded test binding dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 7.45
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.07
SARS-CoV-2-Y91C, torch.Size([320]), 9.300833333333332

Loaded test expression dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Siz

In [6]:
class DMSEmbeddedDataset_BE(Dataset):
    """ Binding and Expression DMS Embedded Dataset, multi target. """
    
    def __init__(self, parquet_file:str):
        """
        Load from parquet file into pandas:
        - sequence label ('labels'), 
        - 'embedding',
        - binding target,
        - expression target
        """
        try:
            self.full_df = pd.read_parquet(parquet_file, engine='fastparquet')
        except (FileNotFoundError, pd.errors.ParserError, Exception) as e:
            print(f"Error reading in .parquet file: {parquet_file}\n{e}", file=sys.stderr)
            sys.exit(1)

    def __len__(self) -> int:
        return len(self.full_df)

    def __getitem__(self, idx):
        # label, embedding, binding target, expression target
        return self.full_df['label'][idx], torch.tensor(self.full_df['embedding'][idx]), self.full_df['ACE2-binding_affinity'][idx], self.full_df['RBD_expression'][idx]
    
# Load in the parquets
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
embedded_train_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_train_ESM-CLS-embedded.parquet")
train_parquet_loader = DMSEmbeddedDataset_BE(embedded_train_parquet)
print("Loaded training dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, b_target, e_target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {b_target}, {e_target}")

embedded_test_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_test_ESM-CLS-embedded.parquet")
test_parquet_loader = DMSEmbeddedDataset_BE(embedded_test_parquet)
print("\nLoaded test dataset from parquet:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, b_target, e_target = train_parquet_loader[i]
    print(f"{label}, {embedding.shape}, {b_target}, {e_target}")

Loaded training dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 8.62, 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 9.78, 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 6.0, 7.45
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.6, 8.07
SARS-CoV-2-Y91C, torch.Size([320]), 10.364166666666668, 9.300833333333332

Loaded test dataset from parquet:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 8.62, 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 9.78, 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 6.0, 7.45
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.6, 8.07
SARS-CoV-2-Y91C, torch.Size([320]), 10.364166666666668, 9.300833333333332


### ESM (Amino acid embedding BROKEN)

In [7]:
# Parquet Making
class DMSDataset(Dataset):
    """ DMS virus sequence dataset. """

    def __init__(self, csv_file:str):
        self.df = pd.read_csv(csv_file, header=0, na_filter=False)
        self.max_sequence_length = self.df['sequence'].apply(len).max()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        columns = (
            self.df.iloc[idx]["label"],
            self.df.iloc[idx]["target"],
            self.df.iloc[idx]["sequence"],
            self.df.iloc[idx]["ACE2-binding_affinity"],
            self.df.iloc[idx]["RBD_expression"],
        )
        return columns

class ESM(nn.Module):
    def __init__(self, esm):
        super().__init__()
        self.esm = esm

    def forward(self, tokenized_seqs):
        with torch.set_grad_enabled(self.training):  # Enable gradients, managed by model.eval() or model.train() in epoch_iteration
            esm_last_hidden_state = self.esm(**tokenized_seqs).last_hidden_state # shape: (batch_size, sequence_length, embedding_dim)
            esm_aa_embedding = esm_last_hidden_state[:, 1:-1, :] # Amino Acid-level representations, [batch_size, sequence_length-2, embedding_dim], excludes 1st and last tokens
        return esm_aa_embedding

def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the ESM model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []

    for _, batch_data in data_iter:
        labels, targets, sequences, binding_scores, expression_scores = batch_data 

        # Add 2 to max_length to account for additional tokens added to beginning and end by ESM
        max_length = dataloader.dataset.max_sequence_length + 2
        tokenized_seqs = tokenizer(sequences, return_tensors='pt', padding='max_length', max_length=max_length).to(device) 
        
        with torch.no_grad():
            embeddings = model(tokenized_seqs) # shape: [batch_size, sequence_len, embedding_dim]

        # Create a DataFrame for the batch
        batch_df = pd.DataFrame({
            "label": labels,
            "target": targets,
            "ACE2-binding_affinity": binding_scores,
            "RBD_expression": expression_scores,
            "embedding": [embedding.flatten().cpu().numpy() for embedding in embeddings]  # Convert tensor to numpy array for each embedding
        })

        batch_dataframes.append(batch_df)

    # Concatenate all batch DataFrames into one
    result_df = pd.concat(batch_dataframes, ignore_index=True)
    
    # Save data to a Parquet file
    save_as = csv_file.replace(".csv", "_ESM-AA-embedded.parquet")
    save_as = save_as.replace("/data/dms", "/data/dms/parquets")
    result_df.to_parquet(save_as, index=False)
    print(f"Data saved to {save_as}")

In [8]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, train_dataloader, device, train_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 1285/1285 [01:30<00:00, 14.16it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_train_ESM-AA-embedded.parquet


In [9]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, test_dataloader, device, test_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 322/322 [00:21<00:00, 15.32it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_test_ESM-AA-embedded.parquet


In [ ]:
embedded_train_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_train_ESM-AA-embedded.parquet")
df = pd.read_parquet(embedded_train_parquet, engine='fastparquet')
df

```
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
Corrupted thrift data at  -328802966 :  13 15
...
```

And the kernel dies.

### NLP Embedding
Currently used as comparison in the paper.

In [2]:
# Parquet Making
class DMSDataset(Dataset):
    """ DMS virus sequence dataset. """

    def __init__(self, csv_file:str):
        self.df = pd.read_csv(csv_file, header=0, na_filter=False)
        self.max_sequence_length = self.df['sequence'].apply(len).max()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        columns = (
            self.df.iloc[idx]["label"],
            self.df.iloc[idx]["target"],
            self.df.iloc[idx]["sequence"],
            self.df.iloc[idx]["ACE2-binding_affinity"],
            self.df.iloc[idx]["RBD_expression"],
        )
        return columns

def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []

    for _, batch_data in data_iter:
        labels, targets, sequences, binding_scores, expression_scores = batch_data 

        tokenized_seqs = tokenizer(sequences).to(device) 
        
        with torch.no_grad():
            embeddings, _ = model(tokenized_seqs) # shape: [batch_size, seq_length embedding_dim]

        # Create a DataFrame for the batch
        batch_df = pd.DataFrame({
            "label": labels,
            "target": targets,
            "ACE2-binding_affinity": binding_scores,
            "RBD_expression": expression_scores,
            "embedding": [embedding.flatten().cpu().numpy() for embedding in embeddings]  # Convert tensor to numpy array for each embedding
        })

        batch_dataframes.append(batch_df)

    # Concatenate all batch DataFrames into one
    result_df = pd.concat(batch_dataframes, ignore_index=True)
    
    # Save data to a Parquet file
    save_as = csv_file.replace(".csv", "_NLP-embedded.parquet")
    save_as = save_as.replace("/data/dms", "/data/dms/parquets")
    result_df.to_parquet(save_as, index=False)
    print(f"Data saved to {save_as}")

In [3]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# NLP input
max_len = 280
embedding_dim = 320
dropout = 0.1
mask_prob=0

model = NLPEmbedding(embedding_dim, max_len, dropout)
tokenizer = ProteinTokenizer(max_len, mask_prob)

# Load NLP weights from BERT model
bert_model_pth = "../../results/run_results/bert_mlm-esm_init/bert_mlm-esm_init-RBD-2024-09-25_20-29/best_saved_model.pth"
saved_state = torch.load(bert_model_pth, map_location=device, weights_only=False)
model_state = saved_state['model_state_dict']
embedding_weights = model_state['bert.embedding.token_embedding.weight']
with torch.no_grad():
    model.token_embedding.weight = nn.Parameter(embedding_weights)

run_model(model, tokenizer, train_dataloader, device, train_csv_file)

100%|| 1285/1285 [00:29<00:00, 43.36it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_train_NLP-embedded.parquet


In [4]:
# Set device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# NLP input
max_len = 280
embedding_dim = 320
dropout = 0.1
mask_prob=0

model = NLPEmbedding(embedding_dim, max_len, dropout)
tokenizer = ProteinTokenizer(max_len, mask_prob)

# Load NLP weights from BERT model
bert_model_pth = "../../results/run_results/bert_mlm-esm_init/bert_mlm-esm_init-RBD-2024-09-25_20-29/best_saved_model.pth"
saved_state = torch.load(bert_model_pth, map_location=device, weights_only=False)
model_state = saved_state['model_state_dict']
embedding_weights = model_state['bert.embedding.token_embedding.weight']
with torch.no_grad():
    model.token_embedding.weight = nn.Parameter(embedding_weights)

run_model(model, tokenizer, test_dataloader, device, test_csv_file)

100%|| 322/322 [00:07<00:00, 42.67it/s]


Data saved to ../../data/dms/parquets/mutation_combined_DMS_OLD_test_NLP-embedded.parquet


: 

In [ ]:
embedded_train_parquet = os.path.join(data_dir, "parquets/mutation_combined_DMS_OLD_train_NLP-embedded.parquet")
df = pd.read_parquet(embedded_train_parquet, engine='fastparquet')
df

```
Corrupted thrift data at  -356028686 :  3 11
Corrupted thrift data at  -356028685 :  10 15
Corrupted thrift data at  -356028682 :  2 11
Corrupted thrift data at  -356028686 :  16 11
Corrupted thrift data at  -356028685 :  23 15
Corrupted thrift data at  -356028682 :  2 11
Corrupted thrift data at  -356028686 :  16 11
Corrupted thrift data at  -356028685 :  23 15
Corrupted thrift data at  -356028682 :  2 11
Corrupted thrift data at  -356028686 :  16 11
Corrupted thrift data at  -356028685 :  23 15
...
```

And the kernel dies.

Looks like this route of saving into parquets is not going to work. The data gets corrupted and the parquets are large if the embedding needs to be flattened to 2D to be saved. We will have to shift to embedding the sequences before passing them to the model (like we are doing now, but not saving the embeddings into a parquet).